In [1]:
import numpy as np
import pandas as pd
#from sklearn.preprocessing import OrdinalEncoder

In [ ]:
# Directory Dataset (HD) Cleaning & Label Mapping
import pandas as pd
import numpy as np

df_hd = pd.read_csv("ipeds_institution_directory_2025.csv")

target_columns_map = {
    "UNITID": "unitid",            
    "INSTNM": "institution_name",   
    "CITY": "city",                 
    "STABBR": "state",              
    "OBEREG": "bea_region",         
    "SECTOR": "sector",             
    "ICLEVEL": "iclevel",           
    "CONTROL": "control",           
    "HLOFFER": "highest_offering",  
    "UGOFFER": "offers_undergrad",  
    "GROFFER": "offers_grad",       
    "DEGGRANT": "degree_granting"   
}

# Safeguard: handle case variations in raw IPEDS headers
col_map_case_insensitive = {col.upper(): col for col in df_hd.columns}
available_cols = [col_map_case_insensitive[col] for col in target_columns_map.keys() if col in col_map_case_insensitive]

df_hd_clean = df_hd[available_cols].copy()
rename_dict = {col_map_case_insensitive[orig]: clean for orig, clean in target_columns_map.items() if orig in col_map_case_insensitive}
df_hd_clean.rename(columns=rename_dict, inplace=True)


df_hd_clean = df_hd_clean[df_hd_clean["sector"] != 0].copy()

# Standardize Binary Indicators (1 = Yes, 2/Negative = 0)
hd_binary_cols = ["offers_undergrad", "offers_grad", "degree_granting"]
for col in hd_binary_cols:
    if col in df_hd_clean.columns:
        df_hd_clean[col] = df_hd_clean[col].apply(lambda x: 1 if x == 1 else 0)


# BEA Region Mapping
bea_map = {
    1: "New England",
    2: "Mid East",
    3: "Great Lakes",
    4: "Plains",
    5: "Southeast",
    6: "Southwest",
    7: "Rocky Mountains",
    8: "Far West",
    0: "Outlying Areas",
    9: "Outlying Areas"
}
df_hd_clean["bea_region"] = df_hd_clean["bea_region"].map(bea_map)

# Sector Mapping (9 active operational categories)
sector_map = {
    1: "Public, 4-year or above",
    2: "Public, 2-year",
    3: "Public, less-than-2-year",
    4: "Private non-profit, 4-year or above",
    5: "Private non-profit, 2-year",
    6: "Private non-profit, less-than-2-year",
    7: "Private for-profit, 4-year or above",
    8: "Private for-profit, 2-year",
    9: "Private for-profit, less-than-2-year"
}
df_hd_clean["sector"] = df_hd_clean["sector"].map(sector_map)

# Institutional Level (ICLEVEL) Mapping
iclevel_map = {
    1: "4-year or above",
    2: "2-year",
    3: "Less than 2-year"
}
df_hd_clean["iclevel"] = df_hd_clean["iclevel"].map(iclevel_map)

# Institutional Control Mapping
control_map = {
    1: "Public",
    2: "Private non-profit",
    3: "Private for-profit"
}
df_hd_clean["control"] = df_hd_clean["control"].map(control_map)

# Highest Offering Mapping
hloffer_map = {
    1: "Certificate (< 1 year)",
    2: "Certificate (1 to 2 years)",
    3: "Associate's degree",
    4: "Certificate (2 to 4 years)",
    5: "Bachelor's degree",
    6: "Postbaccalaureate certificate",
    7: "Master's degree",
    8: "Post-master's certificate",
    9: "Doctor's degree"
}
df_hd_clean["highest_offering"] = df_hd_clean["highest_offering"].map(hloffer_map)

# Save HD dataset
df_hd_clean.to_csv("ipeds_clean_hd.csv", index=False)
print(f"Cell 1 Complete - Clean HD Dataset Shape: {df_hd_clean.shape}")

Cell 1 Complete - Clean HD Dataset Shape: (5914, 12)


In [ ]:
# Institutional Characteristics (IC) Cleaning
df_ic = pd.read_csv("ipeds_institution_characteristics_2025.csv")

ic_columns_map = {
    "UNITID": "unitid",
    "PEO1ISTR": "offers_occupational_prog",
    "PEO2ISTR": "offers_academic_prog",
    "PEO3ISTR": "offers_continuing_ed",
    "PEO4ISTR": "offers_recreational_prog",
    "PEO5ISTR": "offers_remedial_instruction",
    "PEO6ISTR": "offers_secondary_prog",
    "CALSYS": "calendar_system",
    "FT_UG": "enrolls_ft_undergrad",
    "PT_UG": "enrolls_pt_undergrad",
    "DOCAC": "uses_clock_hours",
    "SLO5": "provides_job_placement",
    "SLO6": "provides_employment_services",
    "DISTNCED": "offers_distance_courses",
    "DSTNCED1": "offers_cert_under_12wks",
    "DSTNCED2": "offers_cert_12to52wks"
}

# Safeguard column name casing across raw headers
col_map_ic = {col.upper(): col for col in df_ic.columns}
available_ic_cols = [col_map_ic[col] for col in ic_columns_map.keys() if col in col_map_ic]

df_ic_clean = df_ic[available_ic_cols].copy()
rename_ic_dict = {col_map_ic[orig]: clean for orig, clean in ic_columns_map.items() if orig in col_map_ic}
df_ic_clean.rename(columns=rename_ic_dict, inplace=True)

# Recode All Indicator Features to Standard Binary (1 / 0)
# Keeping 0s so non-occupational providers serve as negative controls
indicator_cols = [c for c in df_ic_clean.columns if c not in ["unitid", "calendar_system"]]

for col in indicator_cols:
    df_ic_clean[col] = df_ic_clean[col].apply(lambda x: 1 if x == 1 else 0)

# Save IC dataset
df_ic_clean.to_csv("ipeds_clean_ic.csv", index=False)
print(f"Cell 2 Complete - Clean IC Dataset Shape: {df_ic_clean.shape}")

Cell 2 Complete - Clean IC Dataset Shape: (5905, 15)


In [ ]:
# Merging & Workforce Feature Creation

# 1. Merge HD and IC on unitid
df_merged = pd.merge(df_hd_clean, df_ic_clean, on="unitid", how="inner")

# Feature A: Sub-baccalaureate institution flag (2-year or less-than-2-year)
df_merged["is_sub_baccalaureate"] = df_merged["iclevel"].isin(["2-year", "Less than 2-year"]).astype(int)

# Feature B: Short-term occupational certificate provider flag
cert_cols = [c for c in ["offers_cert_under_12wks", "offers_cert_12to52wks"] if c in df_merged.columns]
if cert_cols:
    df_merged["is_short_term_cert_provider"] = (df_merged[cert_cols].sum(axis=1) > 0).astype(int)
else:
    df_merged["is_short_term_cert_provider"] = 0

# Feature C: Comprehensive career & employment support flag
if "provides_job_placement" in df_merged.columns and "provides_employment_services" in df_merged.columns:
    df_merged["has_career_support_infrastructure"] = (
        (df_merged["provides_job_placement"] == 1) & 
        (df_merged["provides_employment_services"] == 1)
    ).astype(int)

df_merged.to_csv("ipeds_workforce_pell_dataset.csv", index=False)

print(f"Cell 3 Complete - Final Dataset Shape: {df_merged.shape}")
df_merged.head()

Cell 3 Complete - Final Dataset Shape: (5834, 29)


,unitid,institution_name,city,state,bea_region,sector,iclevel,control,highest_offering,offers_undergrad,...,enrolls_ft_undergrad,enrolls_pt_undergrad,provides_job_placement,provides_employment_services,offers_distance_courses,offers_cert_under_12wks,offers_cert_12to52wks,is_sub_baccalaureate,is_short_term_cert_provider,has_career_support_infrastructure
0,100654,Alabama A & M University,Normal,AL,Southeast,"Public, 4-year or above",4-year or above,Public,Doctor's degree,1,...,1,1,1,1,0,1,1,0,1,1
1,100663,University of Alabama at Birmingham,Birmingham,AL,Southeast,"Public, 4-year or above",4-year or above,Public,Doctor's degree,1,...,1,1,1,1,0,1,1,0,1,1
2,100690,Amridge University,Montgomery,AL,Southeast,"Public, 2-year",4-year or above,Private non-profit,Doctor's degree,1,...,1,1,0,0,1,1,1,0,1,0
3,100706,University of Alabama in Huntsville,Huntsville,AL,Southeast,"Public, 4-year or above",4-year or above,Public,Doctor's degree,1,...,1,1,1,1,0,1,1,0,1,1
4,100724,Alabama State University,Montgomery,AL,Southeast,"Public, 4-year or above",4-year or above,Public,Doctor's degree,1,...,1,1,1,1,0,1,1,0,1,1


In [ ]:
feature_summary = pd.DataFrame({
    "Data Type": df_merged.dtypes,
    "Unique Values": df_merged.nunique(),
    "Sample Value": df_merged.iloc[0]
})

feature_summary

,Data Type,Unique Values,Sample Value
unitid,int64,5834,100654
institution_name,str,5735,Alabama A & M University
city,str,2254,Normal
state,str,59,AL
bea_region,str,9,Southeast
sector,str,9,"Public, 4-year or above"
iclevel,str,3,4-year or above
control,str,3,Public
highest_offering,str,9,Doctor's degree
offers_undergrad,int64,2,1


In [ ]:
# Feature Encoding & ML Matrix Preparation
from sklearn.preprocessing import OrdinalEncoder

df = pd.read_csv("ipeds_workforce_pell_dataset.csv")

unitids = df["unitid"]
X = df.drop(columns=["unitid", "institution_name", "city", "state"], errors="ignore").copy()

X["iclevel"] = X["iclevel"].fillna("Unknown")
X["highest_offering"] = X["highest_offering"].fillna("Unknown")

# Map ICLEVEL
iclevel_order = ["Less than 2-year", "2-year", "4-year or above"]
iclevel_encoder = OrdinalEncoder(
    categories=[iclevel_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)
X["iclevel_encoded"] = iclevel_encoder.fit_transform(X[["iclevel"]])

# Map Highest Offering
hloffer_order = [
    "Certificate (< 1 year)",
    "Certificate (1 to 2 years)",
    "Associate's degree",
    "Certificate (2 to 4 years)",
    "Bachelor's degree",
    "Postbaccalaureate certificate",
    "Master's degree",
    "Post-master's certificate",
    "Doctor's degree"
]
hloffer_encoder = OrdinalEncoder(
    categories=[hloffer_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)
X["highest_offering_encoded"] = hloffer_encoder.fit_transform(X[["highest_offering"]])

# Drop original ordinal string columns
X.drop(columns=["iclevel", "highest_offering"], inplace=True)

# One-Hot Encoding (Nominal Unordered Variables)
nominal_cols = [c for c in ["bea_region", "control", "sector"] if c in X.columns]
X_encoded = pd.get_dummies(X, columns=nominal_cols, drop_first=False, dtype=int)


X_encoded.to_csv("ipeds_ml_feature_matrix.csv", index=False)

print(f"Cell 4 Complete - Model Feature Matrix Shape: {X_encoded.shape}")
print(f"Total Predictor Features: {X_encoded.shape[1]}")
X_encoded.head()

Cell 4 Complete - Model Feature Matrix Shape: (5834, 43)
Total Predictor Features: 43


,offers_undergrad,offers_grad,degree_granting,offers_occupational_prog,offers_academic_prog,offers_continuing_ed,offers_recreational_prog,offers_remedial_instruction,offers_secondary_prog,calendar_system,...,control_Public,"sector_Private for-profit, 2-year","sector_Private for-profit, 4-year or above","sector_Private for-profit, less-than-2-year","sector_Private non-profit, 2-year","sector_Private non-profit, 4-year or above","sector_Private non-profit, less-than-2-year","sector_Public, 2-year","sector_Public, 4-year or above","sector_Public, less-than-2-year"
0,1,1,1,0,1,0,0,0,0,1,...,1,0,0,0,0,0,0,0,1,0
1,1,1,1,0,1,1,0,0,0,1,...,1,0,0,0,0,0,0,0,1,0
2,1,1,1,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
3,1,1,1,0,1,1,1,0,0,1,...,1,0,0,0,0,0,0,0,1,0
4,1,1,1,1,1,0,0,0,0,1,...,1,0,0,0,0,0,0,0,1,0
